<a href="https://colab.research.google.com/github/jarekwan/praca_inzynierska/blob/main/predict_traditional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/ml_project', exist_ok=True)
print("folder ready")

import sys
sys.path.append('/content/drive/MyDrive/ml_project')

In [ ]:
%%writefile /content/drive/MyDrive/ml_project/predict_traditional.py
# -*- coding: utf-8 -*-
import os
import pickle
import numpy as np
import pandas as pd

target_dir = "/content/drive/MyDrive/ml_project"

model_path = os.path.join(target_dir, "traditional_trained.pkl")
y_train_path = os.path.join(target_dir, "y_train.pkl")
y_pred_path = os.path.join(target_dir, "y_pred_traditional.pkl")


# ---------------------------------------------------------
# predict_traditional
# ---------------------------------------------------------
def predict_traditional():

    if not os.path.exists(model_path):
        raise FileNotFoundError("traditional_trained.pkl not found")

    if not os.path.exists(y_train_path):
        raise FileNotFoundError("y_train.pkl not found")

    # load trained model
    with open(model_path, "rb") as f:
        model = pickle.load(f)

    # load y_train (needed for naive and moving average)
    y_train = pd.read_pickle(y_train_path)
    y_series = pd.Series(y_train).dropna()

    # h = forecast horizon (length of y_test)
    y_test_path = os.path.join(target_dir, "y_test.pkl")
    if not os.path.exists(y_test_path):
        raise FileNotFoundError("y_test.pkl not found")
    y_test = pd.read_pickle(y_test_path)
    h = len(y_test)

    # ---------------------------------------------------------
    # naive
    # ---------------------------------------------------------
    if isinstance(model, dict) and model.get("type") == "naive":
        last_value = y_series.iloc[-1]
        y_pred = np.array([last_value] * h)

    # ---------------------------------------------------------
    # moving average
    # ---------------------------------------------------------
    elif isinstance(model, dict) and model.get("type") == "moving_average":
        w = model["window"]
        ma_value = y_series.iloc[-w:].mean()
        y_pred = np.array([ma_value] * h)

    # ---------------------------------------------------------
    # ses
    # ---------------------------------------------------------
    elif str(type(model)).endswith("SimpleExpSmoothingResults'>"):
        y_pred = model.forecast(h)
        y_pred = np.array(y_pred)

    # ---------------------------------------------------------
    # arima
    # ---------------------------------------------------------
    elif hasattr(model, "forecast"):
        y_pred = model.forecast(steps=h)
        y_pred = np.array(y_pred)

    else:
        raise ValueError("unknown traditional model type")

    # save predictions
    with open(y_pred_path, "wb") as f:
        pickle.dump(y_pred, f)

    print("saved:", y_pred_path)
    print("traditional predictions generated")

    return y_pred
